# 03 Silver para Gold

Este notebook monta a camada gold a partir das 8 tabelas da silver. A gold segue um **modelo estrela**: uma tabela fato com um chamado por linha e cinco dimensões em volta dela.

| Tabela | Grão | Origem |
|---|---|---|
| `dim_motivo` | um motivo | `silver.tb_base_motivos` |
| `dim_canal` | um canal | `silver.tb_canais` |
| `dim_atendente` | um atendente, mais o membro "Não identificado" | `silver.tb_base_atendentes` |
| `dim_cliente` | um cliente | `silver.tb_clientes` |
| `dim_data` | um dia do período do case | gerada no notebook |
| `fato_chamados` | um chamado | `silver.tb_chamados`, `tb_chamados_hora`, `tb_custos`, `tb_pesquisa_satisfacao` |

**Surrogate keys:** cada dimensão tem uma chave artificial (`sk_*`) e a fato aponta só para essas chaves. Isso tira o CPF (`id_cliente`) da fato, permite representar atendente inválido como "Não identificado" em vez de nulo e deixa a gold independente do formato dos ids dos sistemas de origem.

**Idempotência:** todas as tabelas são gravadas com `overwrite`. Rodar de novo sobre a mesma silver gera o mesmo conteúdo.

**Antes de rodar:** a silver precisa estar completa (notebook `02_Bronze_To_Silver`). As dimensões são gravadas antes da fato, porque a fato lê as surrogate keys já gravadas.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from itertools import chain

# Padronização dos nomes dos schemas (mesmo padrão da bronze e da silver)
CATALOG = "vcredit_medallion"
SILVER_SCHEMA = f"{CATALOG}.silver"
GOLD_SCHEMA = f"{CATALOG}.gold"

# Símbolos padronizados de log
OK = "✅"
FAIL = "❌"
WARN = "⚠️"

# Período do case: janeiro de 2025 a julho de 2026 (o mesmo usado na silver)
DATA_INICIO = "2025-01-01"
DATA_FIM = "2026-07-31"

print(f"SILVER_SCHEMA: {SILVER_SCHEMA}")
print(f"GOLD_SCHEMA: {GOLD_SCHEMA}")
print(f"Período: {DATA_INICIO} a {DATA_FIM}")

## Funções auxiliares

- `adicionar_sk`: cria a surrogate key com `row_number()` ordenado pela chave natural. Como a ordenação é fixa, a mesma silver gera sempre as mesmas chaves.
- `gravar`: grava a tabela na gold como Delta (`overwrite` + `overwriteSchema`) e acrescenta `_gold_atualizado_em`, a coluna de controle com o momento da gravação. O `_` indica que é metadado, não dado de negócio.
- `verificar`: mesmo padrão da silver. Loga OK, WARN ou FAIL; quando a validação é crítica e falha, interrompe o notebook.

In [0]:
def adicionar_sk(df, nome_sk, chave_natural):
    """Cria surrogate key sequencial ordenada pela chave natural, como primeira coluna."""
    janela = Window.orderBy(chave_natural)
    return df.withColumn(nome_sk, F.row_number().over(janela).cast("bigint")).select(nome_sk, *df.columns)


def gravar(df, nome_tabela):
    """Grava na gold como Delta, com a coluna de controle _gold_atualizado_em."""
    tabela = f"{GOLD_SCHEMA}.{nome_tabela}"
    (
        df.withColumn("_gold_atualizado_em", F.current_timestamp())
        .write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(tabela)
    )
    print(f"{OK} {tabela}: {spark.table(tabela).count()} linhas")


def verificar(descricao, qtd, critico=False):
    """Loga OK/WARN/FAIL. Se critico e qtd > 0, interrompe o notebook."""
    if qtd == 0:
        print(f"{OK} {descricao}: 0")
    elif critico:
        print(f"{FAIL} {descricao}: {qtd}")
        raise Exception(f"Validação crítica falhou: {descricao} ({qtd})")
    else:
        print(f"{WARN} {descricao}: {qtd}")


def validar_dimensao(df, nome_sk, chave_natural, linhas_esperadas):
    """Validações comuns a toda dimensão: contagem, sk única e chave natural única."""
    verificar(f"{nome_sk}: linhas diferentes do esperado", abs(df.count() - linhas_esperadas), critico=True)
    verificar(f"{nome_sk} nula", df.filter(F.col(nome_sk).isNull()).count(), critico=True)
    verificar(f"{nome_sk} duplicada", df.count() - df.select(nome_sk).distinct().count(), critico=True)
    com_chave = df.filter(F.col(chave_natural).isNotNull())
    verificar(f"{chave_natural} duplicado",
              com_chave.count() - com_chave.select(chave_natural).distinct().count(), critico=True)


print(f"{OK} funções carregadas")

## dim_motivo e dim_canal

**Origem:** `silver.tb_base_motivos` e `silver.tb_canais`
**Destino:** `gold.dim_motivo` e `gold.dim_canal`

**Transformações:**
- Seleção só das colunas usadas em análise. `nome_motivo` e `nome_canal` viram `motivo` e `canal`, que é como aparecem nos gráficos.
- `sk_motivo` e `sk_canal` criadas a partir de `id_motivo` e `id_canal`.
- A coluna de controle da silver (`_ingerido_em`) não entra; a gold tem a própria (`_gold_atualizado_em`).

**Validações (críticas):**
- Mesma quantidade de linhas da silver (14 motivos, 4 canais).
- Surrogate key sem nulos e sem duplicatas.
- Chave natural sem duplicatas.

In [0]:
silver_motivos = spark.table(f"{SILVER_SCHEMA}.tb_base_motivos")
silver_canais = spark.table(f"{SILVER_SCHEMA}.tb_canais")

dim_motivo = adicionar_sk(
    silver_motivos.select("id_motivo", F.col("nome_motivo").alias("motivo"), "categoria", "criticidade"),
    "sk_motivo", "id_motivo",
)

dim_canal = adicionar_sk(
    silver_canais.select("id_canal", F.col("nome_canal").alias("canal"), "tipo_canal"),
    "sk_canal", "id_canal",
)

# ---- Validações ----
validar_dimensao(dim_motivo, "sk_motivo", "id_motivo", silver_motivos.count())
validar_dimensao(dim_canal, "sk_canal", "id_canal", silver_canais.count())

# ---- Gravação ----
gravar(dim_motivo, "dim_motivo")
gravar(dim_canal, "dim_canal")

## dim_atendente e dim_cliente

**Origem:** `silver.tb_base_atendentes` e `silver.tb_clientes`
**Destino:** `gold.dim_atendente` e `gold.dim_cliente`

**Transformações:**
- `sk_atendente` e `sk_cliente` criadas a partir de `id_atendente` e `id_cliente`.
- **Membro "Não identificado"** na `dim_atendente`, com `sk_atendente = -1`. Os chamados com atendente inválido na origem (ids 0, 99, 999) apontam para ele em vez de ficarem com chave nula, que faria esses chamados sumirem de qualquer análise por atendente ou nível.
- **Dados pessoais fora da gold:** `nome` e `email` do cliente não entram, porque não servem para nenhuma análise agregada. Continuam na silver para quem precisar identificar um cliente pelo `id_cliente`.
- O `id_cliente` (CPF) fica só na `dim_cliente`, como ponte com a silver. A fato usa apenas a `sk_cliente`, então o acesso ao CPF pode ser restringido a uma tabela.
- Idade e região mantêm os nulos que a silver definiu (idade fora de 18 a 100, região fora das 5 regiões).

**Validações (críticas):**
- `dim_atendente` com as linhas da silver mais o membro "Não identificado" (25 + 1).
- `dim_cliente` com as mesmas linhas da silver (47.705).
- Surrogate key sem nulos e sem duplicatas; chave natural sem duplicatas.

In [0]:
silver_atendentes = spark.table(f"{SILVER_SCHEMA}.tb_base_atendentes")
silver_clientes = spark.table(f"{SILVER_SCHEMA}.tb_clientes")

dim_atendente = adicionar_sk(
    silver_atendentes.select(
        "id_atendente", "nome_atendente", "nivel_atendimento",
        "data_admissao", "data_desligamento", "status_atendente",
    ),
    "sk_atendente", "id_atendente",
)

# Membro desconhecido: as demais colunas ficam nulas
nao_identificado = spark.createDataFrame(
    [(-1, "Não identificado")], "sk_atendente bigint, nome_atendente string"
)
dim_atendente = dim_atendente.unionByName(nao_identificado, allowMissingColumns=True)

dim_cliente = adicionar_sk(
    silver_clientes.select("id_cliente", "regiao", "idade", "data_cadastro", "dia_vencimento_fatura"),
    "sk_cliente", "id_cliente",
)

# ---- Validações ----
validar_dimensao(dim_atendente, "sk_atendente", "id_atendente", silver_atendentes.count() + 1)
validar_dimensao(dim_cliente, "sk_cliente", "id_cliente", silver_clientes.count())
verificar("membro Não identificado ausente",
          int(dim_atendente.filter(F.col("sk_atendente") == -1).count() != 1), critico=True)

# ---- Gravação ----
gravar(dim_atendente, "dim_atendente")
gravar(dim_cliente, "dim_cliente")

## dim_data

**Origem:** gerada no notebook (não vem da silver)
**Destino:** `gold.dim_data`

**Por que existe:** concentra os atributos de calendário (ano, mês, trimestre, dia da semana) num lugar só, e qualquer análise da fato faz join pela data em vez de recalcular. Como tem todos os dias do período, um gráfico por dia não pula datas sem chamado.

**Transformações:**
- Uma linha por dia de `DATA_INICIO` a `DATA_FIM`, com `sequence` (lista de datas) e `explode` (uma linha por data).
- `sk_data` no formato `yyyyMMdd` (ex.: `20250101`), padrão de mercado para chave de data: legível e ordenável.
- Nomes de mês e dia da semana em português, por dicionário, porque o `date_format` do Spark devolve os nomes em inglês.
- `dia_semana` segue o padrão do Spark: 1 = domingo, 7 = sábado.

**Validações (críticas):**
- 577 dias (365 de 2025 mais 212 de janeiro a julho de 2026).
- `sk_data` e `data` sem nulos e sem duplicatas.

In [0]:
nomes_mes = {1: "janeiro", 2: "fevereiro", 3: "março", 4: "abril", 5: "maio", 6: "junho",
             7: "julho", 8: "agosto", 9: "setembro", 10: "outubro", 11: "novembro", 12: "dezembro"}
nomes_dia = {1: "domingo", 2: "segunda", 3: "terça", 4: "quarta", 5: "quinta", 6: "sexta", 7: "sábado"}

# Transforma os dicionários em mapas do Spark para consultar pelo número
mapa_mes = F.create_map([F.lit(x) for x in chain(*nomes_mes.items())])
mapa_dia = F.create_map([F.lit(x) for x in chain(*nomes_dia.items())])

dim_data = (
    spark.sql(f"""
        SELECT explode(sequence(to_date('{DATA_INICIO}'), to_date('{DATA_FIM}'), interval 1 day)) AS data
    """)
    .withColumn("sk_data", F.date_format("data", "yyyyMMdd").cast("int"))
    .withColumn("ano", F.year("data"))
    .withColumn("mes", F.month("data"))
    .withColumn("nome_mes", mapa_mes[F.month("data")])
    .withColumn("ano_mes", F.date_format("data", "yyyy-MM"))
    .withColumn("trimestre", F.quarter("data"))
    .withColumn("dia_semana", F.dayofweek("data"))
    .withColumn("nome_dia_semana", mapa_dia[F.dayofweek("data")])
    .withColumn("fim_de_semana", F.dayofweek("data").isin(1, 7))
)
dim_data = dim_data.select("sk_data", *[c for c in dim_data.columns if c != "sk_data"])

# ---- Validações ----
dias_esperados = spark.sql(f"SELECT datediff(to_date('{DATA_FIM}'), to_date('{DATA_INICIO}')) + 1 AS n").first()["n"]
validar_dimensao(dim_data, "sk_data", "data", dias_esperados)

# ---- Gravação ----
gravar(dim_data, "dim_data")

## Agregação de custos e pesquisas por chamado

**Origem:** `silver.tb_custos` e `silver.tb_pesquisa_satisfacao`
**Destino:** DataFrames intermediários (não são gravados)

**Por que agregar antes do join:** o grão da fato é um chamado por linha, mas um chamado pode ter vários lançamentos de custo e mais de uma pesquisa (802 chamados têm mais de uma). Se custos e pesquisas fossem juntados direto, o chamado se repetiria uma vez para cada lançamento ou pesquisa, e qualquer contagem ou soma da gold sairia inflada sem gerar erro.

**Transformações:**
- Custos: soma por chamado (`custo_total`), quantidade de lançamentos e quantidade de lançamentos negativos.
- Os custos negativos (271 lançamentos) entram na soma com sinal, porque a regra deles (estorno ou erro) ainda não foi definida. `qtd_custos_negativos` permite separar esses chamados depois sem refazer a gold.
- Pesquisas: média das notas válidas por chamado (`nota_media`) e quantidade de pesquisas. Notas ausentes ou fora da escala de 1 a 5 ficam fora da média.

**Validações (críticas):**
- A soma dos custos agregados é igual à soma da silver.
- Cada chamado aparece uma vez só nos dois DataFrames, sem `id_chamado` nulo.

In [0]:
custos_por_chamado = (
    spark.table(f"{SILVER_SCHEMA}.tb_custos")
    .groupBy("id_chamado")
    .agg(
        F.sum("custo").alias("custo_total"),
        F.count("*").alias("qtd_lancamentos_custo"),
        F.sum(F.col("custo_negativo").cast("int")).alias("qtd_custos_negativos"),
    )
)

pesquisas_por_chamado = (
    spark.table(f"{SILVER_SCHEMA}.tb_pesquisa_satisfacao")
    .groupBy("id_chamado")
    .agg(
        # when sem otherwise devolve nulo para as notas inválidas, e o avg ignora nulos
        F.avg(F.when(F.col("nota_valida"), F.col("nota_atendimento"))).alias("nota_media"),
        F.count("*").alias("qtd_pesquisas"),
    )
)

# ---- Validações ----
soma_silver = spark.table(f"{SILVER_SCHEMA}.tb_custos").agg(F.sum("custo")).first()[0]
soma_agregada = custos_por_chamado.agg(F.sum("custo_total")).first()[0]
print(f"soma dos custos: silver = {soma_silver} | agregada = {soma_agregada}")
verificar("diferença entre a soma da silver e a agregada", int(soma_silver != soma_agregada), critico=True)

for nome, df in [("custos", custos_por_chamado), ("pesquisas", pesquisas_por_chamado)]:
    linhas = df.count()
    verificar(f"{nome}: id_chamado repetido após agregação",
              linhas - df.select("id_chamado").distinct().count(), critico=True)
    verificar(f"{nome}: id_chamado nulo", df.filter(F.col("id_chamado").isNull()).count(), critico=True)
    print(f"{nome}: {linhas} chamados")

## fato_chamados

**Origem:** `silver.tb_chamados`, `silver.tb_chamados_hora` e os agregados de custos e pesquisas
**Destino:** `gold.fato_chamados`

**Grão:** um chamado por linha.

**Decisões:**
- **Base:** `tb_chamados`. Todos os joins são `left` a partir dela e todas as chaves da direita são únicas, então nenhum chamado se perde nem se repete.
- **Cliente:** vem do `tb_chamados_hora`. 1.330 `id_cliente` do sistema de chamados não existem na base de clientes, e o `tb_chamados_hora` tem o cliente correto para esses casos. Quando o do `tb_chamados_hora` está vazio, usa o de `tb_chamados`.
- **Horários e tempos:** vêm do `tb_chamados_hora`, a fonte oficial de horários definida na silver. Os tempos usados são os tratados (`tempo_*_segundos`), nulos quando a ordem dos horários é inválida.
- **Período:** só chamados abertos de janeiro de 2025 a julho de 2026 (`abertura_no_periodo`), o mesmo período da `dim_data`. Hoje saem 198 chamados, todos com abertura em 2027.
- **Custo sem lançamento fica nulo**, que é diferente de custo zero. As quantidades (`qtd_*`) sem registro ficam 0.
- **Surrogate keys:** depois de montada com as chaves naturais, a fato troca cada uma pela `sk_*` da dimensão. Atendente sem par recebe `-1` ("Não identificado"). Nenhuma chave natural fica na fato além de `id_chamado`, que é a identificação do próprio chamado.

**Validações (críticas):**
- Os joins não mudam a quantidade de chamados.
- A troca pelas surrogate keys não muda a quantidade de linhas.
- `id_chamado` único e fato não vazia.
- `sk_cliente`, `sk_motivo`, `sk_canal` e `sk_data` sem nulos.

In [0]:
chamados = spark.table(f"{SILVER_SCHEMA}.tb_chamados").select(
    "id_chamado",
    F.col("id_cliente").alias("id_cliente_chamados"),
    "id_atendente",
    "id_motivo",
    "id_canal",
    "resolvido",
)

hora = spark.table(f"{SILVER_SCHEMA}.tb_chamados_hora").select(
    "id_chamado",
    # Texto vazio ou "null" vira nulo de verdade
    F.when(~F.lower(F.trim("id_cliente")).isin("", "null"), F.trim("id_cliente")).alias("id_cliente_hora"),
    "hora_abertura_chamado",
    "abertura_no_periodo",
    "ordem_temporal_invalida",
    "tempo_espera_segundos",
    "tempo_atendimento_segundos",
    "tempo_total_segundos",
)

base = (
    chamados
    .join(hora, "id_chamado", "left")
    .join(custos_por_chamado, "id_chamado", "left")
    .join(pesquisas_por_chamado, "id_chamado", "left")
)

# Etapa 1: fato com as chaves naturais, já filtrada pelo período do case
fato_natural = base.filter(F.col("abertura_no_periodo") == True).select(
    "id_chamado",
    F.coalesce("id_cliente_hora", "id_cliente_chamados").alias("id_cliente"),
    "id_atendente",
    "id_motivo",
    "id_canal",
    F.to_date("hora_abertura_chamado").alias("data_abertura"),
    F.hour("hora_abertura_chamado").alias("hora_do_dia"),
    "hora_abertura_chamado",
    "resolvido",
    "tempo_espera_segundos",
    "tempo_atendimento_segundos",
    "tempo_total_segundos",
    "ordem_temporal_invalida",
    "custo_total",
    F.coalesce("qtd_lancamentos_custo", F.lit(0)).alias("qtd_lancamentos_custo"),
    F.coalesce("qtd_custos_negativos", F.lit(0)).alias("qtd_custos_negativos"),
    "nota_media",
    F.coalesce("qtd_pesquisas", F.lit(0)).alias("qtd_pesquisas"),
)

# Etapa 2: troca das chaves naturais pelas surrogate keys gravadas nas dimensões
d_cli = spark.table(f"{GOLD_SCHEMA}.dim_cliente").select("id_cliente", "sk_cliente")
d_ate = (spark.table(f"{GOLD_SCHEMA}.dim_atendente")
         .filter(F.col("sk_atendente") != -1)
         .select("id_atendente", "sk_atendente"))
d_mot = spark.table(f"{GOLD_SCHEMA}.dim_motivo").select("id_motivo", "sk_motivo")
d_can = spark.table(f"{GOLD_SCHEMA}.dim_canal").select("id_canal", "sk_canal")

fato_chamados = (
    fato_natural
    .join(d_cli, "id_cliente", "left")
    .join(d_ate, "id_atendente", "left")
    .join(d_mot, "id_motivo", "left")
    .join(d_can, "id_canal", "left")
    .withColumn("sk_atendente", F.coalesce("sk_atendente", F.lit(-1).cast("bigint")))
    .withColumn("sk_data", F.date_format("data_abertura", "yyyyMMdd").cast("int"))
    .select(
        "id_chamado",
        "sk_cliente", "sk_atendente", "sk_motivo", "sk_canal", "sk_data",
        "hora_do_dia", "hora_abertura_chamado",
        "resolvido",
        "tempo_espera_segundos", "tempo_atendimento_segundos", "tempo_total_segundos",
        "ordem_temporal_invalida",
        "custo_total", "qtd_lancamentos_custo", "qtd_custos_negativos",
        "nota_media", "qtd_pesquisas",
    )
)

# ---- Validações ----
total_chamados = chamados.count()
total_base = base.count()
sem_horario = base.filter(F.col("hora_abertura_chamado").isNull()).count()
fora_periodo = base.filter(F.col("hora_abertura_chamado").isNotNull() & ~F.col("abertura_no_periodo")).count()
total_natural = fato_natural.count()
total_fato = fato_chamados.count()

print(f"chamados na silver: {total_chamados} | após joins: {total_base}")
print(f"sem horário: {sem_horario} | fora do período: {fora_periodo} | na fato: {total_fato}")

verificar("chamados perdidos ou duplicados nos joins", abs(total_base - total_chamados), critico=True)
verificar("linhas perdidas ou duplicadas na troca por surrogate keys", abs(total_fato - total_natural), critico=True)
verificar("id_chamado repetido na fato",
          total_fato - fato_chamados.select("id_chamado").distinct().count(), critico=True)
verificar("fato vazia", int(total_fato == 0), critico=True)
for sk in ["sk_cliente", "sk_motivo", "sk_canal", "sk_data"]:
    verificar(f"{sk} nula na fato", fato_chamados.filter(F.col(sk).isNull()).count(), critico=True)
verificar("chamados com atendente não identificado (mantidos)",
          fato_chamados.filter(F.col("sk_atendente") == -1).count())

# ---- Gravação ----
gravar(fato_chamados, "fato_chamados")

## Integridade referencial

Confere, nas tabelas já gravadas, se toda chave da fato existe na dimensão correspondente. Um chamado com chave sem par sumiria de qualquer análise que use aquela dimensão, sem gerar erro. O `left_anti` devolve as linhas da fato que não encontram par na dimensão.

**Resultado esperado:** zero em todas as chaves. Os chamados com atendente inválido aparecem como `sk_atendente = -1`, que existe na `dim_atendente`.

In [0]:
fato = spark.table(f"{GOLD_SCHEMA}.fato_chamados")

checagens = [
    ("sk_cliente", "dim_cliente"),
    ("sk_atendente", "dim_atendente"),
    ("sk_motivo", "dim_motivo"),
    ("sk_canal", "dim_canal"),
    ("sk_data", "dim_data"),
]

for chave, dim in checagens:
    dim_df = spark.table(f"{GOLD_SCHEMA}.{dim}").select(chave)
    verificar(f"{chave} nula na fato", fato.filter(F.col(chave).isNull()).count(), critico=True)
    verificar(f"{chave} sem par na {dim}", fato.join(dim_df, chave, "left_anti").count(), critico=True)

print(f"{OK} gold validada")